# Mean Reversion Factor Research

This notebook demonstrates how to build and evaluate **mean reversion** factors, including:

1. Z-score distance from moving average
2. Volatility normalization
3. Cross-sectional processing (`cs_rank`, `cs_zscore`, `cs_winsorize`, `cs_demean`)
4. Comparing raw vs. processed signals
5. Market-neutral vs. un-neutralized backtesting
6. Advanced statistical operators (`ts_autocorr`, `ts_kurtosis`, `ts_skewness`)

**Prerequisites**: `pip install factorium`

## 1. Setup & Data Loading

In [ ]:
from factorium import BinanceDataLoader, ResearchSession
from factorium.factors import FactorAnalyzer

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
SYMBOLS = [
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT",
    "DOGEUSDT", "ADAUSDT", "AVAXUSDT", "DOTUSDT", "CAKEUSDT",
]

loader = BinanceDataLoader()

agg = loader.load_aggbar(
    symbols=SYMBOLS,
    data_type="aggTrades",
    market_type="futures",
    futures_type="um",
    days=30,
    bar_type="time",
    interval=60_000,
)

print(f"Loaded {len(agg):,} bars for {len(agg.symbols)} symbols")

session = ResearchSession(agg, default_frequency="1min")

## 2. Building Mean Reversion Factors

The core idea: prices that deviate far from their moving average tend to **revert back**. We measure this deviation as a z-score.

$$\text{MeanRev}(t) = \frac{P(t) - \text{MA}(t, w)}{\sigma(t, w)}$$

In [ ]:
close = agg["close"]
volume = agg["volume"]

# --- Mean Reversion Factor (z-score from 60-period MA) ---
ma_60 = close.ts_mean(60)
std_60 = close.ts_std(60)
mean_rev_raw = (close - ma_60) / std_60
mean_rev_raw.name = "mean_rev_raw"

# For mean reversion, we go OPPOSITE to the deviation:
# high z-score → price is above MA → expect reversion down → SHORT
# So we negate the signal
mean_rev = -mean_rev_raw
mean_rev.name = "mean_rev"

print(f"Raw mean reversion factor: {len(mean_rev):,} rows")
mean_rev.to_pandas().head()

### 2.1 Volatility-Normalized Mean Reversion

Normalizing by realized volatility ensures the signal is comparable across assets with different volatility levels.

In [ ]:
# Compute returns and rolling volatility
returns = close.ts_delta(1) / close.ts_shift(1)
realized_vol = returns.ts_std(60)

# Volatility-normalized mean reversion
mean_rev_vol_norm = mean_rev / realized_vol
mean_rev_vol_norm.name = "mean_rev_vol_norm"

print(f"Vol-normalized mean reversion: {len(mean_rev_vol_norm):,} rows")

## 3. Cross-Sectional Processing

Before using a factor as a trading signal, we typically apply **cross-sectional transformations** to normalize across assets at each point in time.

In [ ]:
# Cross-sectional rank: maps to [0, 1] across symbols at each timestamp
signal_ranked = mean_rev.cs_rank()
signal_ranked.name = "mean_rev_ranked"

# Cross-sectional z-score: standardize across symbols
signal_zscore = mean_rev.cs_zscore()
signal_zscore.name = "mean_rev_zscore"

# Winsorize: clip extreme values (2.5th and 97.5th percentiles)
signal_winsorized = mean_rev.cs_winsorize(limits=0.025)
signal_winsorized.name = "mean_rev_winsorized"

# Demean: remove cross-sectional mean
signal_demeaned = mean_rev.cs_demean()
signal_demeaned.name = "mean_rev_demeaned"

print("Cross-sectional transformations applied:")
print(f"  Ranked:     [{signal_ranked.data['factor'].min():.3f}, {signal_ranked.data['factor'].max():.3f}]")
print(f"  Z-scored:   mean={signal_zscore.data['factor'].mean():.6f}")
print(f"  Demeaned:   mean={signal_demeaned.data['factor'].mean():.6f}")

## 4. Comparing Raw vs. Processed Signals

Let's see how cross-sectional processing affects the factor's predictive power (IC).

In [ ]:
factors_to_compare = {
    "Raw Mean Rev": mean_rev,
    "Vol-Normalized": mean_rev_vol_norm,
    "CS Ranked": signal_ranked,
    "CS Z-Score": signal_zscore,
    "CS Winsorized": signal_winsorized,
}

comparison_results = []
for name, factor in factors_to_compare.items():
    analysis = session.analyze(factor, periods=1)
    ic_stats = analysis.ic_summary.get(1, {})
    comparison_results.append({
        "Factor": name,
        "Mean IC": ic_stats.get("mean_ic", np.nan),
        "IC Std": ic_stats.get("ic_std", np.nan),
        "IC IR": ic_stats.get("ic_ir", np.nan),
    })

comparison_df = pd.DataFrame(comparison_results).set_index("Factor")
print("IC Comparison — Raw vs. Processed Signals:")
comparison_df

In [ ]:
# Visualize IC comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

comparison_df["Mean IC"].plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Mean IC")
axes[0].axvline(x=0, color="gray", linestyle="--")

comparison_df["IC IR"].plot.barh(ax=axes[1], color="coral")
axes[1].set_title("IC IR (Information Ratio)")
axes[1].axvline(x=0, color="gray", linestyle="--")

plt.suptitle("Mean Reversion Factor — Signal Processing Comparison", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Backtest Comparison: Market-Neutral vs. Un-Neutralized

Compare how the factor performs under different portfolio construction approaches.

In [ ]:
# Use the best-performing signal variant for backtesting
best_signal = signal_ranked

# Market-neutral backtest
result_neutral = session.backtest(best_signal, neutralization="market")

# Un-neutralized (long-only) backtest
result_long = session.backtest(best_signal, neutralization="none")

# Compare metrics
metrics_comparison = pd.DataFrame({
    "Market Neutral": result_neutral.metrics,
    "Long Only": result_long.metrics,
}).T

key_metrics = ["total_return", "annual_return", "sharpe_ratio", "max_drawdown", "sortino_ratio"]
available_metrics = [m for m in key_metrics if m in metrics_comparison.columns]
metrics_comparison[available_metrics]

In [ ]:
# Plot equity curves side-by-side
fig, ax = plt.subplots(figsize=(14, 6))

eq_neutral = result_neutral.equity_curve.to_pandas()
eq_neutral["ts"] = pd.to_datetime(eq_neutral["start_time"], unit="ms")
ax.plot(eq_neutral["ts"], eq_neutral["equity"], label="Market Neutral", linewidth=1.5)

eq_long = result_long.equity_curve.to_pandas()
eq_long["ts"] = pd.to_datetime(eq_long["start_time"], unit="ms")
ax.plot(eq_long["ts"], eq_long["equity"], label="Long Only", linewidth=1.5)

ax.set_xlabel("Date")
ax.set_ylabel("Equity")
ax.set_title("Mean Reversion — Equity Curves")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Advanced Statistical Operators

Factorium provides advanced time-series operators that are useful for characterizing market microstructure and regime analysis.

In [ ]:
returns = close.ts_delta(1) / close.ts_shift(1)

# Autocorrelation: measures serial dependence in returns
# Positive autocorrelation → trending, Negative → mean reverting
autocorr = returns.ts_autocorr(60, lag=1)
autocorr.name = "return_autocorr"

# Kurtosis: measures tail heaviness
# High kurtosis → more extreme events (fat tails)
kurtosis = returns.ts_kurtosis(60)
kurtosis.name = "return_kurtosis"

# Skewness: measures asymmetry of the return distribution
# Negative skew → more downside risk
skewness = returns.ts_skewness(60)
skewness.name = "return_skewness"

print("Advanced statistical factors computed:")
for f in [autocorr, kurtosis, skewness]:
    print(f"  {f.name}: {len(f):,} rows")

In [ ]:
# Visualize autocorrelation for a few symbols
autocorr.plot.plot_timeseries(symbols=["BTCUSDT", "ETHUSDT", "SOLUSDT"])
plt.axhline(y=0, color="red", linestyle="--", alpha=0.5)
plt.title("Rolling Return Autocorrelation (60-period window, lag=1)")
plt.ylabel("Autocorrelation")
plt.show()

In [ ]:
# Quick IC check on autocorrelation as a factor
# (negative autocorrelation → mean reverting → potential signal)
autocorr_signal = (-autocorr).cs_rank()
autocorr_signal.name = "neg_autocorr_rank"

autocorr_analysis = session.analyze(autocorr_signal, periods=1)
print("Autocorrelation as factor:")
print(autocorr_analysis.ic_summary)

## 7. Summary

In this notebook we:
- Built a **mean reversion factor** using z-score distance from a moving average
- Applied **volatility normalization** to make the signal stable across assets
- Demonstrated **cross-sectional processing** methods (`cs_rank`, `cs_zscore`, `cs_winsorize`, `cs_demean`)
- Compared IC across different signal processing methods
- Ran backtests under **market-neutral** and **long-only** regimes
- Explored **advanced statistical operators** for regime characterization

**Key takeaway:** Cross-sectional normalization often improves factor stability (IC IR), even if it doesn't always increase mean IC.